In [1]:
import pandas as pd
import sys
sys.path.append('..')

# Import existing project functions
from functions.BAG_buildings_API import fetch_buildings_from_BAG
from functions.BAG_addresses_API import enrich_buildings_with_addresses
from functions.process_buildings import process_and_visualize_buildings
from functions.TNO_API import fetch_residential_heat_demand

In [12]:
# Configuration (from run_analysis.py)
BAG_API_KEY = 'l7c0673beb4a3f46e8a0caa164dc7b8397'

# Area code
AREA = '4342' 
year=2020

# Bounding box for Multatulibuurt (lon, lat pairs)
bbox_coords = [
    (4.3588444390090535, 51.98977145529007),
    (4.363727554070601, 51.99104189404924),
    (4.3599960417556245, 51.997399351063684),
    (4.356750677717929, 51.996732491653034),
    (4.354965562453528, 51.995617668217804),
    (4.358146528659458, 51.98999163382852),
    (4.3588444390090535, 51.98977145529007)
]

# Convert to bounding box [min_lon, min_lat, max_lon, max_lat]
lons = [coord[0] for coord in bbox_coords]
lats = [coord[1] for coord in bbox_coords]
bounding_box = [min(lons), min(lats), max(lons), max(lats)]

print(f"Area code: {AREA}")
print(f"Bounding box: {bounding_box}")

Area code: 4342
Bounding box: [4.354965562453528, 51.98977145529007, 4.363727554070601, 51.997399351063684]


In [3]:
# Fetch all buildings in the bounding box
print("Fetching buildings from BAG API...")
all_buildings = fetch_buildings_from_BAG(bounding_box, BAG_API_KEY)
print(f"Fetched {len(all_buildings)} buildings")

# Enrich with addresses
print("\nEnriching buildings with address data...")
addresses_dict = enrich_buildings_with_addresses(all_buildings, BAG_API_KEY)
print(f"Enriched {len(addresses_dict)} buildings with addresses")

Fetching buildings from BAG API...
Fetched 885 buildings

Enriching buildings with address data...
Enriched 848 buildings with addresses


In [4]:
# Process buildings using existing function
print("Processing building data...")
buildings_df = process_and_visualize_buildings(
    all_buildings, 
    addresses_dict, 
    mode='export'  # Don't create visualizations, just process
)

Processing building data...


In [13]:
# Fetch residential heat demand from TNO API
print(f"Fetching heat demand data from TNO API for area {AREA}...")
try:
    residential_heat_demand = fetch_residential_heat_demand(AREA, year)
    
except Exception as e:
    print(f"\n❌ Error fetching heat demand data: {e}")
    print("\nThe TNO API may be unavailable. Consider using the surrogate data approach instead.")
    residential_heat_demand = None

Fetching heat demand data from TNO API for area 4342...


In [14]:
if residential_heat_demand is not None:
    combined_output = f'combined_building_heat_data_area_{AREA}.csv'
    residential_heat_demand.to_csv(combined_output, index=False)
    print(f"✓ Combined dataset saved to: {combined_output}")
else:
    print("❌ Cannot save data - heat demand fetch failed")

✓ Combined dataset saved to: combined_building_heat_data_area_4342.csv


## Optional: Quick Visualization

In [14]:
import folium
from pyproj import Transformer

if residential_heat_demand is not None and 'combined_df' in locals():
    print("Creating interactive heat demand map...")
    
    # Convert coordinates from RD (EPSG:28992) to WGS84 (EPSG:4326)
    transformer = Transformer.from_crs("EPSG:28992", "EPSG:4326", always_xy=True)
    combined_df[['lon_wgs84', 'lat_wgs84']] = combined_df.apply(
        lambda row: pd.Series(transformer.transform(row['lon'], row['lat'])),
        axis=1
    )
    
    # Filter out buildings without coordinates
    map_df = combined_df[combined_df['lon_wgs84'].notna() & combined_df['lat_wgs84'].notna()].copy()
    print(f"Mapping {len(map_df)} buildings with coordinates")
    
    # Calculate map center
    center_lat = map_df['lat_wgs84'].mean()
    center_lon = map_df['lon_wgs84'].mean()
    
    # Create Folium map
    heat_map = folium.Map(location=[center_lat, center_lon], zoom_start=15, tiles="OpenStreetMap")
    
    # Normalize heat demand for color scale
    max_demand = map_df['Peak heat demand (kW)'].max()
    min_demand = map_df['Peak heat demand (kW)'].min()
    
    # Add circle markers for each building
    for idx, row in map_df.iterrows():
        demand = row['Peak heat demand (kW)']
        normalized = (demand - min_demand) / (max_demand - min_demand) if max_demand > min_demand else 0.5
        
        # Color from light yellow (low demand) to dark red (high demand)
        red = int(255)
        green = int(255 * (1 - normalized * 0.8))
        blue = int(255 * (1 - normalized))
        color = f'#{red:02x}{green:02x}{blue:02x}'
        
        # Scale circle radius based on heat demand
        radius = max(5, min(20, demand / (max_demand / 20)))  # 5-20 pixels
        
        # Create popup with building information
        address = row.get('addresses', 'N/A')
        popup_html = f"""
        <div style="font-family: Arial; font-size: 12px;">
            <b>Building ID:</b> {row['id']}<br>
            <b>Address:</b> {address}<br>
            <b>Peak Heat Demand:</b> {row['Peak heat demand (kW)']:.2f} kW<br>
            <b>Annual Demand (Warmtevraag):</b> {row['Warmtevraag']:.0f}<br>
            <b>Year Built:</b> {row.get('year_construction', 'N/A')}<br>
        </div>
        """
        
        folium.CircleMarker(
            location=[row['lat_wgs84'], row['lon_wgs84']],
            radius=radius,
            popup=folium.Popup(popup_html, max_width=300),
            color='#333333',
            weight=1,
            fill=True,
            fillColor=color,
            fillOpacity=0.7
        ).add_to(heat_map)
    
    # Add a legend
    legend_html = f'''
    <div style="position: fixed; 
         bottom: 50px; right: 50px; width: 220px; height: 140px; 
         background-color: white; border:2px solid grey; z-index:9999; 
         font-size:14px; padding: 10px">
         <p style="margin: 0; font-weight: bold;">Peak Heat Demand (kW)</p>
         <p style="margin: 5px 0;"><span style="background-color: #FFFF00; padding: 2px 10px;">●</span> Low ({min_demand:.1f})</p>
         <p style="margin: 5px 0;"><span style="background-color: #FF8800; padding: 2px 10px;">●</span> Medium</p>
         <p style="margin: 5px 0;"><span style="background-color: #FF0000; padding: 2px 10px;">●</span> High ({max_demand:.1f})</p>
         <p style="margin: 10px 0 0 0; font-size: 11px; color: #666;">Circle size = heat demand</p>
    </div>
    '''
    heat_map.get_root().html.add_child(folium.Element(legend_html))
    
    # Save map
    map_file = f'heat_demand_map_area_{AREA}.html'
    heat_map.save(map_file)
    print(f"\n✓ Interactive heat demand map saved to: {map_file}")
    print(f"Open this file in your browser to view the map!")
else:
    print("❌ Cannot create map - heat demand data not available")

Creating interactive heat demand map...
Mapping 373 buildings with coordinates

✓ Interactive heat demand map saved to: heat_demand_map_area_4316.html
Open this file in your browser to view the map!
